# 🦅 Fine-tuning Llama-PLLuM-8B-chat — QLoRA

**Model:** `CYFRAGOVPL/Llama-PLLuM-8B-chat`  
**Licencja:** Llama 3.1 (open, dozwolony użytek komercyjny)  
**Metoda:** QLoRA (4-bit) via Unsloth  
**GPU:** T4 15GB 
**Czas treningu:** ~5 min (1 epoka, mały dataset)

### Przed startem:
- `Kaggle` → `Settings` → `Accelerator` → **T4 GPU x 2**

## 1. Instalacja

In [1]:
%%capture
!pip install unsloth
!pip install trl accelerate peft datasets bitsandbytes

## 2. Sprawdzenie GPU

In [ ]:
!nvidia-smi

## 3. Ładowanie modelu + QLoRA

> **Uwaga:** Pobieranie modelu 8B zajmie ~2–3 min. VRAM po załadowaniu: ok. 6–7 GB.

In [ ]:
import unsloth 
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="CYFRAGOVPL/Llama-PLLuM-8B-chat",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Adaptery LoRA — target_modules identyczne jak dla Llama 3.1
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # rank — im wyższy, tym więcej parametrów do trenowania
    lora_alpha=16,
    # lora_dropout=0.05,
    lora_dropout=0.00,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # oszczędza VRAM
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"VRAM zajęte: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Dataset

Przykładowe dane — **zamień na swoje** w formacie `{instruction, input, output}`

> **Ważne:** PLLuM używa formatu **Llama 3** (`<|start_header_id|>`)


In [ ]:
from datasets import Dataset

# ===== TWOJE DANE — edytuj tutaj =====
data = [
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest machine learning?",
        "output": "Machine learning to dziedzina AI, w której modele uczą się na danych bez jawnego programowania."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest fine-tuning?",
        "output": "Fine-tuning to dostosowanie pretrenowanego modelu do konkretnego zadania poprzez dalsze trenowanie na małym zbiorze danych."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest LoRA?",
        "output": "LoRA (Low-Rank Adaptation) to metoda fine-tuningu, która trenuje tylko małe adaptery zamiast całego modelu — oszczędza VRAM i czas."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest tokenizer?",
        "output": "Tokenizer to algorytm zamieniający tekst na liczby (tokeny) zrozumiałe dla modelu językowego."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest GPU?",
        "output": "GPU to procesor graficzny — ma tysiące małych rdzeni, idealnych do równoległych obliczeń macierzowych potrzebnych w AI."
    },
]
# ===== KONIEC DANYCH =====

def format_prompt(example):
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n" 
        f"{example['instruction']}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['input']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}
# Tworzenie datasetu
#     → Dataset.from_list()   # zamienia listę na obiekt HuggingFace Dataset
#     → .map(format_prompt)   # wywołuje format_prompt() na każdym przykładzie
dataset = Dataset.from_list(data).map(format_prompt)
print(f"Przykładów w datasecie: {len(dataset)}")
print("\nPrzykład:")
print(dataset[0]["text"])


## 5. Trening

> **Uwaga dot. batch_size:** Zmniejszamy z 2 → 1 (model 8B zajmuje więcej VRAM niż 1.7B).  
> Gradient accumulation 8 kompensuje — efektywny batch = 8 (taki sam jak wcześniej).

In [ ]:
from trl import SFTTrainer, SFTConfig 

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=512,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="./output",
        optim="adamw_8bit",
        save_strategy="no",
        report_to="none",
    ),
)

print("Start treningu...")
trainer.train()
print("Trening zakończony!")

## 6. Test modelu po fine-tuningu

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, instruction="Odpowiedz krótko po polsku."):
    # Format Llama 3 — BEZ odpowiedzi asystenta (model ma ją wygenerować)
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{instruction}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        return response.split("assistant")[-1].strip()
    return response.strip()

print(ask("Co to jest LoRA?"))
print("---")
print(ask("Wyjaśnij czym jest PLLuM."))